# FAA Wildlife Strike Analysis### Analyzing 30+ years of aviation wildlife strike data**Dataset:** FAA Wildlife Strike Database (Official US Government Data)  **Tools:** Python, Pandas, Seaborn, Plotly  **Goal:** Identify high-risk flight phases, dangerous species, and airport-level risk patterns

## Problem StatementAnalyze FAA wildlife strike records to identify when, where, and how strikes occur most often, then use those patterns to prioritize aviation safety and wildlife mitigation efforts.

In [ ]:
# STEP 1 - IMPORTSimport osimport pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsimport plotly.express as pximport warningswarnings.filterwarnings('ignore')sns.set_theme(style='darkgrid')plt.rcParams['figure.figsize'] = (12, 6)print("All libraries imported successfully!")

In [ ]:
# STEP 2 - LOAD DATAos.chdir(r'E:\Arein\Projects\FAA_Wildlife_Strike_Analysis')df = pd.read_csv('wildlife.csv', low_memory=False)print(f"Raw data loaded! Rows: {len(df):,}")

## Data Cleaning

In [ ]:
# STEP 3 - CLEAN DAMAGE LEVEL CODESdamage_map = {    'N': 'NONE',    'M': 'MINOR',    'M?': 'MINOR',    'S': 'SUBSTANTIAL',    'D': 'DESTROYED',    'UNKNOWN': 'UNKNOWN'}df['DAMAGE_LEVEL'] = df['DAMAGE_LEVEL'].map(damage_map).fillna('UNKNOWN')df.to_csv('wildlife_clean.csv', index=False)print(df['DAMAGE_LEVEL'].value_counts())print('Cleaned data saved to wildlife_clean.csv')

## Exploratory Data Analysis

In [ ]:
# EDA QUESTIONSyearly = df['INCIDENT_YEAR'].value_counts().sort_index()phase = df['PHASE_OF_FLIGHT'].value_counts()species = df['SPECIES'].value_counts().head(10)states = df[df['STATE'] != 'UNKNOWN']['STATE'].value_counts().head(10)time_day = df[df['TIME_OF_DAY'] != 'UNKNOWN']['TIME_OF_DAY'].value_counts()damage = df[df['DAMAGE_LEVEL'] != 'UNKNOWN']['DAMAGE_LEVEL'].value_counts()ac_class = df['AC_CLASS'].value_counts()print('=== STRIKES PER YEAR ===')print(yearly.tail(10))print(f"\nPeak Year : {yearly.idxmax()} ({yearly.max():,} strikes)")print(f"First Year: {yearly.index.min()}")print(f"Last Year : {yearly.index.max()}")print('\n=== STRIKES BY PHASE OF FLIGHT ===')print(phase)print('\n=== TOP 10 SPECIES ===')print(species)print('\n=== TOP 10 STATES ===')print(states)print('\n=== STRIKES BY TIME OF DAY ===')print(time_day)print('\n=== DAMAGE LEVEL DISTRIBUTION ===')print(damage)print('\n=== STRIKES BY AIRCRAFT CLASS ===')print(ac_class)

## Visualizations

In [ ]:
# CHART 1 - Yearly trendyearly = df['INCIDENT_YEAR'].value_counts().sort_index()yearly = yearly[yearly.index <= 2025]plt.figure(figsize=(14, 6))plt.plot(yearly.index, yearly.values, color='#2196F3', linewidth=2.5, marker='o', markersize=4)plt.fill_between(yearly.index, yearly.values, alpha=0.2, color='#2196F3')plt.title('Wildlife Strikes Per Year (1990-2025)', fontsize=16, fontweight='bold', pad=15)plt.xlabel('Year', fontsize=12)plt.ylabel('Number of Strikes', fontsize=12)plt.xticks(rotation=45)peak_year = yearly.idxmax()peak_val = yearly.max()plt.annotate(f'Peak: {peak_val:,}', xy=(peak_year, peak_val), xytext=(peak_year-5, peak_val-2000), arrowprops=dict(arrowstyle='->', color='red'), fontsize=11, color='red')plt.tight_layout()plt.savefig('chart1_yearly_trend.png', dpi=150, bbox_inches='tight')plt.show()# CHART 2 - Phase of flightphase = df[df['PHASE_OF_FLIGHT'] != 'UNKNOWN']['PHASE_OF_FLIGHT'].value_counts()plt.figure(figsize=(12, 6))bars = plt.barh(phase.index, phase.values, color=sns.color_palette('Blues_r', len(phase)))plt.title('Wildlife Strikes by Phase of Flight', fontsize=16, fontweight='bold', pad=15)plt.xlabel('Number of Strikes', fontsize=12)plt.ylabel('Phase of Flight', fontsize=12)for bar, val in zip(bars, phase.values):    plt.text(val + 200, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=10)plt.tight_layout()plt.savefig('chart2_phase_of_flight.png', dpi=150, bbox_inches='tight')plt.show()# CHART 3 - Top 10 speciesspecies = df[~df['SPECIES'].str.contains('UNKNOWN', na=False)]['SPECIES'].value_counts().head(10)plt.figure(figsize=(12, 6))bars = plt.barh(species.index[::-1], species.values[::-1], color=sns.color_palette('Greens_r', 10))plt.title('Top 10 Wildlife Species Involved in Strikes', fontsize=16, fontweight='bold', pad=15)plt.xlabel('Number of Strikes', fontsize=12)for bar, val in zip(bars, species.values[::-1]):    plt.text(val + 100, bar.get_y() + bar.get_height()/2, f'{val:,}', va='center', fontsize=10)plt.tight_layout()plt.savefig('chart3_top_species.png', dpi=150, bbox_inches='tight')plt.show()# CHART 4 - Top 10 statesstates = df[df['STATE'] != 'UNKNOWN']['STATE'].value_counts().head(10)plt.figure(figsize=(12, 6))colors = ['#FF5252' if i == 0 else '#2196F3' for i in range(len(states))]bars = plt.bar(states.index, states.values, color=colors)plt.title('Top 10 States by Wildlife Strikes', fontsize=16, fontweight='bold', pad=15)plt.xlabel('State', fontsize=12)plt.ylabel('Number of Strikes', fontsize=12)for bar, val in zip(bars, states.values):    plt.text(bar.get_x() + bar.get_width()/2, val + 100, f'{val:,}', ha='center', fontsize=9)plt.tight_layout()plt.savefig('chart4_top_states.png', dpi=150, bbox_inches='tight')plt.show()# CHART 5 - Damage level donut chartdamage = df[df['DAMAGE_LEVEL'] != 'UNKNOWN']['DAMAGE_LEVEL'].value_counts()colors = ['#4CAF50', '#FFC107', '#FF9800', '#F44336']explode = [0.02, 0.05, 0.05, 0.08]plt.figure(figsize=(10, 7))wedges, texts, autotexts = plt.pie(damage.values, labels=damage.index, autopct='%1.1f%%', colors=colors, explode=explode, startangle=90, wedgeprops=dict(width=0.6))for text in autotexts:    text.set_fontsize(11)    text.set_fontweight('bold')plt.title('Damage Level Distribution', fontsize=13, fontweight='bold', pad=15)plt.tight_layout()plt.savefig('chart5_damage_distribution.png', dpi=150, bbox_inches='tight')plt.show()# CHART 6 - Time of daytime_order = ['DAY', 'NIGHT', 'DUSK', 'DAWN']time_day = df[df['TIME_OF_DAY'] != 'UNKNOWN']['TIME_OF_DAY'].value_counts().reindex(time_order)colors = ['#FFD700', '#1A237E', '#FF7043', '#FF8F00']plt.figure(figsize=(10, 6))bars = plt.bar(time_day.index, time_day.values, color=colors, width=0.5)plt.title('Wildlife Strikes by Time of Day', fontsize=16, fontweight='bold', pad=15)plt.xlabel('Time of Day', fontsize=12)plt.ylabel('Number of Strikes', fontsize=12)for bar, val in zip(bars, time_day.values):    plt.text(bar.get_x() + bar.get_width()/2, val + 300, f'{val:,}', ha='center', fontsize=11, fontweight='bold')plt.tight_layout()plt.savefig('chart6_time_of_day.png', dpi=150, bbox_inches='tight')plt.show()# CHART 7 - Monthly strike heatmap by year (last 10 years)df_recent = df[df['INCIDENT_YEAR'] >= 2015]heatmap_data = df_recent.groupby(['INCIDENT_YEAR', 'INCIDENT_MONTH']).size().unstack(fill_value=0)heatmap_data = heatmap_data.reindex(columns=range(1, 13), fill_value=0)heatmap_data.columns = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']plt.figure(figsize=(14, 7))sns.heatmap(heatmap_data, annot=True, fmt='g', cmap='YlOrRd', linewidths=0.5, annot_kws={'size': 9})plt.title('Monthly Strike Heatmap (2015-2025)', fontsize=16, fontweight='bold', pad=15)plt.xlabel('Month', fontsize=12)plt.ylabel('Year', fontsize=12)plt.tight_layout()plt.savefig('chart7_monthly_heatmap.png', dpi=150, bbox_inches='tight')plt.show()

## Key Insights & Findings### Based on analysis of 342,201 FAA wildlife strike records (1990-2025)

In [ ]:
# INSIGHT SUMMARY - All key findings in one placeprint('=' * 60)print('   FAA WILDLIFE STRIKE ANALYSIS - KEY INSIGHTS')print('=' * 60)yearly = df[df['INCIDENT_YEAR'] <= 2025]['INCIDENT_YEAR'].value_counts().sort_index()growth = ((yearly[2025] - yearly[2000]) / yearly[2000] * 100).round(1)print(f'''INSIGHT 1 - RAPID GROWTHWildlife strikes grew by {growth}% from 2000 to 2025.From {yearly[2000]:,} strikes in 2000 to {yearly[2025]:,} in 2025.Driven by increased air traffic + improved reporting systems.''')phase = df[df['PHASE_OF_FLIGHT'] != 'UNKNOWN']['PHASE_OF_FLIGHT'].value_counts()top_phase = phase.index[0]top_phase_pct = (phase[top_phase] / phase.sum() * 100).round(1)print(f'''INSIGHT 2 - APPROACH IS MOST DANGEROUS{top_phase} phase accounts for {top_phase_pct}% of all known-phase strikes.Combined with LANDING ROLL & TAKE-OFF, low-altitude phases = over 75% of all strikes.''')species = df[~df['SPECIES'].str.contains('UNKNOWN', na=False)]['SPECIES'].value_counts()print(f'''INSIGHT 3 - SMALL BIRDS ARE BIGGEST THREATTop identified species: {species.index[0]} ({species.iloc[0]:,} strikes)Mourning Doves, Barn Swallows & Killdeer dominate strike records.Small birds are dangerous due to their large flocks near airports.''')states = df[df['STATE'] != 'UNKNOWN']['STATE'].value_counts()print(f'''INSIGHT 4 - TEXAS IS HIGHEST RISK STATETexas leads with {states['TX']:,} strikes - followed by Florida ({states['FL']:,}) and California ({states['CA']:,}).Warm climates = year-round bird activity near busy airports.''')time_day = df[df['TIME_OF_DAY'] != 'UNKNOWN']['TIME_OF_DAY'].value_counts()day_pct = (time_day['DAY'] / time_day.sum() * 100).round(1)print(f'''INSIGHT 5 - DAYTIME IS MOST COMMON, NIGHT IS MORE DANGEROUS{day_pct}% of strikes happen during daytime (pilots can see wildlife).Night strikes are fewer but more severe - pilots can't spot animals.Dawn & Dusk are high-risk transition periods for bird activity.''')damage = df[df['DAMAGE_LEVEL'] != 'UNKNOWN']['DAMAGE_LEVEL'].value_counts()total_known = damage.sum()substantial = damage.get('SUBSTANTIAL', 0)destroyed = damage.get('DESTROYED', 0)serious_pct = ((substantial + destroyed) / total_known * 100).round(2)print(f'''INSIGHT 6 - SERIOUS DAMAGE IS RARE BUT REALOnly {serious_pct}% of strikes cause substantial damage or destruction.But with {substantial + destroyed:,} serious incidents over 35 years, the safety risk cannot be ignored.{destroyed} aircraft were completely destroyed by wildlife strikes.''')print('=' * 60)

## RecommendationsBased on the analysis, here are 3 actionable safety recommendations:**1. Focus wildlife deterrence during Approach & Landing phases**- Over 75% of strikes happen at low altitude during takeoff/landing- Airports should deploy active bird control during peak flight hours**2. Target High-Risk States with additional resources**- Texas, Florida and California account for highest strike volumes- These airports need year-round wildlife management programs**3. Improve night-time wildlife detection systems**- Night strikes cause disproportionately more damage- Investing in radar-based bird detection systems at major airports would significantly reduce serious incidents

## ConclusionThis analysis of **342,201 FAA wildlife strike records** from 1990-2025 reveals that wildlife strikes are a growing aviation safety concern.Key takeaways:- Strikes have grown **22x** since 1990 driven by air traffic growth- **Approach and landing phases** are highest risk - occurring below 1,000ft- **Small birds** like Mourning Doves cause the most incidents- While **98%+ of strikes cause no serious damage**, the remaining cases represent real safety and economic risk to aviationTargeted wildlife management at high-risk airports during critical flight phases can significantly improve aviation safety outcomes.---*Data Source: FAA National Wildlife Strike Database (Official US Government)*  *Analysis by: Arein Jain*  *Tools: Python, Pandas, Matplotlib, Seaborn*